In [1]:
import os
import re
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
from scipy.sparse import csc_matrix
from math import log
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from qpsolvers import solve_qp

For each data point, please remove the first four lines, i.e. the lines starting with Newsgroup,
document_id, From, Subject.

In [2]:
def process_data(file_path):
    with open(file_path, 'r',encoding= 'latin1') as file:
        lines = file.readlines()
        content = lines[4:]
        content = ' '.join(content)
        content = content.lower()
        content = re.sub(r'[^a-zA-Z\s]', ' ', content)
        content = ' '.join(content.split())
        return content
    
def get_data(root):
    data_by_class = {}
    for folder, dirs, files in os.walk(root):
        class_name = os.path.basename(folder)

        if folder == root:
            continue
        if class_name not in data_by_class:
            data_by_class[class_name] = []

        for file in files:
           file_path = os.path.join(folder,file)
           content = process_data(file_path = file_path)
           data_by_class[class_name].append(content)

    return data_by_class

In [3]:
root_path = "data/20_newsgroups"
data_by_class = get_data(root_path)

Consider the top 300 most frequent words as stop words and remove them from the vocabulary

In [4]:
def get_word_bag(data_by_class):
    word_bag = Counter()
    for class_name, documents in data_by_class.items():
        for doc in documents:
            words = doc.split()  # Split document into words
            word_bag.update(words)  # Update word frequencies
    return word_bag

def remove_stop_words(data_by_class, stop_words):
    filtered_data_by_class = {}
    for class_name, documents in data_by_class.items():
        filtered_data_by_class[class_name] = []
        for doc in documents:
            # Remove stop words from each document
            filtered_doc = ' '.join([word for word in doc.split() if word not in stop_words])
            filtered_data_by_class[class_name].append(filtered_doc)
    return filtered_data_by_class

Split data into two groups, and use half data as training data and the other half as testing data. Note:
split the data of each class into two halves.

In [5]:
def split_data(data_by_class):
    train_data = {}
    test_data = {}
    for class_name, documents in data_by_class.items():
        train_docs, test_docs = train_test_split(documents, test_size=0.5, random_state=42)
        train_data[class_name] = train_docs
        test_data[class_name] = test_docs

    return train_data, test_data

In [6]:
word_bag = get_word_bag(data_by_class)
top_300_words = [word for word, _ in word_bag.most_common(300)] #  Consider the top 300 most frequent words as stop words
features = [word for word, _ in word_bag.most_common(500)] 
vocabulary = [word for word in word_bag.keys() if word in features and word not in top_300_words]

data_by_class = remove_stop_words(data_by_class, top_300_words)

train_data, test_data = split_data(data_by_class)

In [7]:
X_train = []
y_train = []
X_test = []
y_test = []

for y in train_data.keys():
    X_train.extend(train_data[y])
    y_train.extend([y]* len(train_data[y]))

for y in test_data.keys():
    X_test.extend(test_data[y])
    y_test.extend([y]* len(test_data[y]))

Create feature vector for each document using tf-idf method.

In [8]:
tfidf = TfidfVectorizer(vocabulary = vocabulary)
X_train = tfidf.fit_transform(X_train).toarray()
X_test = tfidf.fit_transform(X_test).toarray()
y_train = np.array(y_train)
print(y_train.shape)
# y_test = y_test.toarray()

(9998,)


 Implement the linear soft margin SVM

In [9]:
class SoftMarginSVM:
    def __init__(self, kernel = "linear", C=100):
        self.C = C
        self.kernel = kernel
        self.alpha = None
        self.w = None
        self.b = None
        self.support_vectors = None

    def linear_kernel(self, X, y):
        print("x type",type(X))
        print("y type",type(y))
        n_samples, n_features = X.shape
        yp=np.reshape(y,(1,-1)).repeat(n_features,axis=0)
        print("yp",yp.shape)
        print("yp type",type(yp))
        x1=X.T
        print(x1.shape)
        print("x1 type",type(x1))
        # Xp = np.broadcast_to(yp, x1.shape) 
        
        Xp =x1 * yp
        P = np.dot(Xp.T, Xp)
        return P
    
    def polynomial_kernel(self, X, y):
        n_samples, n_features = X.shape
        P = np.zeros((n_samples, n_samples))
        for i in range(n_samples):
            for j in range(n_samples):
                dot_product = np.dot(X[i],X[j].T)
                # Apply the formula: P_ij = y_i * y_j * (x_i^T x_j)^2
                P[i, j] = y[i] * y[j] * (dot_product ** 2)

        return P
    
    def fit(self, X, y):
        # Number of samples and features
        n_samples, n_features = X.shape

        if self.kernel == "linear":
            P = self.linear_kernel(X,y)
        if self.kernel == "Polynomial":
            P = self.polynomial_kernel(X,y)
        print("P:", P.shape)
        q = -np.ones(n_samples)
        A = y.reshape(1, -1)
        b = np.array([0.0])

        G_std = np.diag(np.ones(n_samples) * -1)  # -I for alpha >= 0
        G_slack = np.identity(n_samples)  # I for alpha <= C
        G = np.vstack((G_std, G_slack))

        h_std = np.zeros(n_samples)  # 0 for alpha >= 0
        h_slack = np.ones(n_samples) * self.C  # C for alpha <= C
        h = np.hstack((h_std, h_slack))

        # Solve the quadratic programming problem
        P = csc_matrix(P)
        G = csc_matrix(G)
        A = csc_matrix(A)
        self.alpha = solve_qp(P, q, G, h, A, b, solver='osqp')

        # Support vectors are where alpha > 0
        support_vector_indices = self.alpha > 1e-5
        self.alpha = self.alpha[support_vector_indices]
        self.support_vectors = X[support_vector_indices]
        self.support_vector_labels = y[support_vector_indices]

        # Weight vector
        self.w = np.sum(self.alpha[:, None] * self.support_vector_labels[:, None] * self.support_vectors, axis=0)

        # Intercept (b)
        self.b = np.mean(
            self.support_vector_labels - np.dot(self.support_vectors, self.w)
        )
       

    def decision_function(self, X):
        return np.dot(X, self.w) + self.b
    def predict(self, X):
        return np.sign(self.decision_function(X))

In [10]:
y_binary = np.where(y_train == y_train[0], 1, -1)
# soft_margin_svm = SoftMarginSVM(kernel= "Polynomial")
soft_margin_svm = SoftMarginSVM()
# soft_margin_svm = SVC()
soft_margin_svm.fit(X_train, y_binary)
y_pred = soft_margin_svm.predict(X_train)
accuracy = accuracy_score(y_binary, y_pred)
print(f"Average Prediction Accuracy: {accuracy * 100:.2f}%")

x type <class 'numpy.ndarray'>
y type <class 'numpy.ndarray'>
yp (200, 9998)
yp type <class 'numpy.ndarray'>
(200, 9998)
x1 type <class 'numpy.ndarray'>
P: (9998, 9998)
Average Prediction Accuracy: 83.42%


 one-vs-all strategy to conduct multi-class SVM

In [11]:
class OneVsAllSvm():
    def __init__ (self, C = 100, kernel = "linear", model = None):
        self.C = C
        self.classifiers = {}
        self.classes = None
        self.model = model
        self.kernel = kernel
    def fit(self, X, y):
        self.classes = np.unique(y)
        for class_label in self.classes:
            y_binary = np.where(y == class_label, 1, -1)
            svm = self.model(kernel= self.kernel, C= self.C)
            svm.fit(X, y_binary)
            self.classifiers[class_label] = svm
    
    def predict(self, X):
        scores = np.column_stack([
            self.classifiers[class_label].decision_function(X)
            for class_label in self.classes
        ])
        return self.classes[np.argmax(scores, axis=1)]

In [12]:
ova_svm = OneVsAllSvm(C = 100, model = SoftMarginSVM)
ova_svm.fit(X_train, y_train)

x type <class 'numpy.ndarray'>
y type <class 'numpy.ndarray'>
yp (200, 9998)
yp type <class 'numpy.ndarray'>
(200, 9998)
x1 type <class 'numpy.ndarray'>
P: (9998, 9998)
x type <class 'numpy.ndarray'>
y type <class 'numpy.ndarray'>
yp (200, 9998)
yp type <class 'numpy.ndarray'>
(200, 9998)
x1 type <class 'numpy.ndarray'>
P: (9998, 9998)
x type <class 'numpy.ndarray'>
y type <class 'numpy.ndarray'>
yp (200, 9998)
yp type <class 'numpy.ndarray'>
(200, 9998)
x1 type <class 'numpy.ndarray'>
P: (9998, 9998)
x type <class 'numpy.ndarray'>
y type <class 'numpy.ndarray'>
yp (200, 9998)
yp type <class 'numpy.ndarray'>
(200, 9998)
x1 type <class 'numpy.ndarray'>
P: (9998, 9998)
x type <class 'numpy.ndarray'>
y type <class 'numpy.ndarray'>
yp (200, 9998)
yp type <class 'numpy.ndarray'>
(200, 9998)
x1 type <class 'numpy.ndarray'>
P: (9998, 9998)
x type <class 'numpy.ndarray'>
y type <class 'numpy.ndarray'>
yp (200, 9998)
yp type <class 'numpy.ndarray'>
(200, 9998)
x1 type <class 'numpy.ndarray'>
P:

In [13]:
print("Linear Soft Margin SVM")
y_pred = ova_svm.predict(X_train)
accuracy = accuracy_score(y_train, y_pred)
print(f"Average Prediction Accuracy on training set is : {accuracy * 100:.2f}%")

y_pred = ova_svm.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Average Prediction Accuracy on testing set is : {accuracy * 100:.2f}%")


Linear Soft Margin SVM
Average Prediction Accuracy on training set is : 40.55%
Average Prediction Accuracy on testing set is : 37.16%


In [14]:
ova_svm = OneVsAllSvm(model = SoftMarginSVM, kernel = "Polynomial", C =100 )
ova_svm.fit(X_train, y_train)

P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)
P: (9998, 9998)


In [15]:
print("Polynomial Kernel Soft Margin SVM")
y_pred =  ova_svm.predict(X_train)
accuracy = accuracy_score(y_train, y_pred)
print(f"Average Prediction Accuracy on training set is : {accuracy * 100:.2f}%")

y_pred = ova_svm.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Average Prediction Accuracy on testing set is : {accuracy * 100:.2f}%")


Polynomial Kernel Soft Margin SVM
Average Prediction Accuracy on training set is : 44.22%
Average Prediction Accuracy on testing set is : 40.45%


Only did 200 vocabulary due to old computer. But accuracy is about 90 on training set for 2000 vocabulary.